# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmad-rind/Flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**I choose Lane 2: Refresh / Content Opportunity Scoring.**

**Why this lane?**

The starter pipeline (the runnable example in this repo) already shows that a learned ranking can meaningfully beat a hand‑tuned baseline. The baseline precision@50 is 0.240, while the random forest reaches 0.740 — that is a **3× improvement** in the number of genuinely high‑priority pages identified among the top 50. This suggests there is real signal in the observable data that can be used to prioritize content reviews more effectively.

Moreover, the lane directly addresses a common business need: **which pages should a content team review first** when resources are limited. The starter data contains clear observable signals (impressions, clicks, position, trends, engagement rates) that can be turned into a ranked queue with reason codes. This makes the project both practical and measurable.

I will build on the starter’s approach but strengthen the label to be **future‑looking**: instead of using the current `trend_direction` bucket as the target, I will define a label based on a future window (e.g., traffic decline or recovery over the next 30 days) and train a model on features from the prior 90 days. This aligns with the “growth / recovery” direction while staying within the refresh‑scoring lane.

The warehouse release (79M daily rows) provides enough history and cross‑client variation to test time‑aware splits and validate that the model generalises to new pages and clients.

**Freestyle?** I am staying within a predefined lane because the lane already fits the problem I find compelling, and the starter code gives me a solid foundation to iterate on.

**Research Question:**
> Given a page’s observable search and engagement signals over the prior 90 days, can we rank pages by their likelihood of needing a content refresh (defined as a future drop or opportunity) so that a content reviewer can examine the most critical pages first?

**Decision:** Which pages appear in the top `K` of a ranked review queue?

**Action:** A content reviewer will inspect the top `K` pages (e.g., 50) and decide whether to update, rewrite, merge, or monitor them. The queue also provides reason codes to speed up the review.

**Cost of a wrong recommendation:**
- **False positive** (recommending a page that doesn’t actually need work): wastes reviewer time, reduces throughput, and may cause a valuable page to be unnecessarily changed.
- **False negative** (missing a page that truly needs work): missed opportunity to improve traffic or engagement, potentially losing visibility over time.

Because reviewer capacity is limited, the metric that matters most is **precision@K** — we want the top recommendations to be as correct as possible.

**Why data or ML can help:**
The starter data shows that simple signals (impressions, position, CTR, trend, age) have predictive power for the current `trend_direction`. A model can combine these signals into a score that outperforms a fixed rule. With a future‑looking label, we can further ensure that the ranking reflects real future outcomes rather than just current status.

In [4]:
import os
import pandas as pd

# Ensure we're in /content
os.chdir('/content')

# Your repo name
repo_name = "Flyrank"  # matches your repo name
csv_path = f"/content/{repo_name}/data/raw/content_refresh_anonymized.csv"

# Clone if not already present
if not os.path.exists(csv_path):
    print("📁 Cloning your repository...")
    !git clone https://github.com/ahmad-rind/Flyrank.git
    # If the repo clones into a different name, adjust here

# Verify and load
if not os.path.exists(csv_path):
    # Try to find it automatically
    found = False
    for root, dirs, files in os.walk('/content'):
        if 'content_refresh_anonymized.csv' in files:
            csv_path = os.path.join(root, 'content_refresh_anonymized.csv')
            found = True
            break
    if not found:
        raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

df = pd.read_csv(csv_path)
print(f"✅ Loaded from: {csv_path}")

# Print the numbers
print(f"Total rows: {len(df):,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")

trend_counts = df['trend_direction'].value_counts()
print("\nTrend direction distribution:")
print(trend_counts)

imp_90d = df['impressions_90d']
print(f"\nImpressions (90d):")
print(f"  Mean: {imp_90d.mean():.1f}")
print(f"  Median: {imp_90d.median():.1f}")
print(f"  Rows with >= 500 impressions: {(imp_90d >= 500).sum():,}")

declining_with_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
print(f"\nDeclining pages with >=100 impressions: {len(declining_with_demand):,}")

✅ Loaded from: /content/Flyrank/data/raw/content_refresh_anonymized.csv
Total rows: 30,000
Unique content IDs: 30,000
Unique clients: 32

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Impressions (90d):
  Mean: 5200.4
  Median: 731.0
  Rows with >= 500 impressions: 16,726

Declining pages with >=100 impressions: 13,152


**I can claim:**
- That a model trained on observable signals (impressions, clicks, position, CTR, engagement, age, etc.) can rank pages by a defined future outcome (e.g., decline or recovery) better than a fixed baseline rule.
- That the ranking can provide actionable reason codes to help a reviewer understand why a page was recommended.
- That the results are observational and reflect patterns in the data, not causal relationships.

**I cannot claim:**
- That refreshing a page will *cause* a recovery — that would require an experiment or a causal design.
- That the model identifies all possible problems; it only works on the signals we choose to include.
- That the recommendations apply universally across all content types or industries; the data is from a specific set of clients and may not generalise.
- That the model predicts exact future metrics; it only produces a relative ranking.

**Leakage precautions:**
- I will define the feature window (prior 90 days) and the target window (next 30 days) such that they do not overlap.
- I will use time‑aware or client‑holdout validation to avoid overfitting.
- I will not use any product decision flags (`health_score`, `action_type`, etc.) as features — only observable signals.

**Before you submit, confirm each line honestly:**

- Every section above is filled — markdown thinking AND the code that backs it.
- The notebook runs top to bottom with no errors (Runtime → Run all).
- No client names, URLs, or private queries anywhere.
- My claims use careful words: observed, measured, directional, decision‑support.
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.